# Circular Shear Viewer Demonstration

This notebook demonstrates `CircularShearViewer` — shear visualization for
circular RC sections using the Orr (2012) shear model.

Three plot methods are available (link angle is fixed at 90° for circular sections):

1. `plot_cot_theta_study` — capacity components vs cot(θ) with λ₁·λ₂ factors
2. `plot_cot_theta_moment_shift_study` — utilization and tension-shift M_add vs cot(θ)
3. `plot_force_cot_theta_contour` — N_Ed vs cot(θ) heatmap

All methods return Plotly figures.

In [14]:
from materials.reinforced_concrete.geometry import (
    create_circular_section,
    create_circular_perimeter_rebars,
)
from materials.reinforced_concrete.materials import ConcreteMaterial, Rebar, ShearRebar
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    CircularSectionCheck,
    ShearLoadCase,
)
from materials.reinforced_concrete.analysis.circular_shear_viewer import CircularShearViewer

## 1. Build a Circular Section and Check

In [15]:
# Section parameters
pile_dia = 600       # mm
cover = 50           # mm to link outer face
link_dia = 16
bar_dia = 20
n_bars = 10

section = create_circular_section(diameter=pile_dia, section_name=f"{pile_dia}mm Pile")

rebar = Rebar(diameter=bar_dia)
rebar_cover = cover + link_dia
section.add_rebar_group(
    create_circular_perimeter_rebars(
        rebar=rebar,
        diameter=pile_dia,
        cover=rebar_cover,
        n_bars=n_bars,
    )
)

concrete = ConcreteMaterial(grade="C30/37")
links = ShearRebar(diameter=link_dia, link_spacing=200, n_legs=2, grade="B500B")

check = CircularSectionCheck(
    section=section,
    concrete=concrete,
    diameter=pile_dia,
    cover=cover,
    shear_reinforcement=links,
)

print(f"Section: {section.section_name}")
print(f"Concrete: {concrete.grade}")
print(f"Links: H{link_dia}@{links.link_spacing} ({links.n_legs}-leg)")
print(f"r_sv: {check.r_sv:.1f} mm")
print(f"\u03bb\u2082: {check.calculate_lambda_2():.3f}")

Section: 600mm Pile
Concrete: C30/37
Links: H16@200.0 (2-leg)
r_sv: 242.0 mm
λ₂: 1.000


## 2. Baseline Shear Check

In [16]:
load_case = ShearLoadCase(V_Ed=400.0, M_Ed=350.0, N_Ed=500.0)
result = check.perform_shear_check(load_case=load_case)
d = result.details

print(f"Utilization: {result.utilization:.3f}")
print(f"Governing: {d.get('governing_mode', 'N/A')}")
print(f"cot(\u03b8): {d['cot_theta']:.3f}")
print(f"\u03bb\u2081: {d['lambda_1']:.4f}, \u03bb\u2082: {d['lambda_2']:.4f}")
print(f"b_w: {d['b_w']:.1f} mm")
print(f"V_Rd,s: {d['V_Rd_s']:.1f} kN, V_Rd,max: {d['V_Rd_max']:.1f} kN")

Utilization: 0.809
Governing: V_Rd_max
cot(θ): 2.500
λ₁: 0.8756, λ₂: 1.0000
b_w: 402.9 mm
V_Rd,s: 645.2 kN, V_Rd,max: 494.7 kN


## 3. Create the Viewer

In [17]:
viewer = CircularShearViewer(check)

## 4. Cot(θ) Capacity Study

Shows `V_Rd,s` (with λ₁·λ₂ efficiency factors) and `V_Rd,max` (with circular b_w)
versus cot(θ). The subtitle reports the computed λ₁, λ₂, and b_w values.

In [18]:
fig_cot = viewer.plot_cot_theta_study(
    load_case=load_case,
    n_points=60,
    show=False,
)
fig_cot

## 5. Cot(θ) Moment-Shift Study

Dual-axis plot showing utilization and tension-shift additional moment (`M_add`)
versus cot(θ). If utilization crosses 1.0, the intercept is marked.

In [19]:
fig_shift = viewer.plot_cot_theta_moment_shift_study(
    load_case=load_case,
    n_points=40,
    show=False,
)
fig_shift

## 6. Axial Force vs Cot(θ) Contour

Heatmap of utilization (or other metrics) over the N_Ed–cot(θ) domain.
For each axial force level the full circular context is recomputed
(σ_cp, λ₁, λ₂, b_w, V_Rd,c, cot limits).

In [20]:
fig_axial = viewer.plot_force_cot_theta_contour(
    load_case=load_case,
    n_axial=40,
    n_cot=40,
    metric="utilization",
    show=False,
)
fig_axial

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\analysis\circular_shear_viewer.py:123: UserWarning:

Effective depth fallback (no compression/tension split): d = 540.0 mm

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:650: UserWarning:

Lever arm fallback to 351.0 mm (0.65d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.



### 6.1 Other Metrics

In [21]:
fig_cap = viewer.plot_force_cot_theta_contour(
    load_case=load_case,
    n_axial=40,
    n_cot=40,
    metric="capacity",
    show=False,
)
fig_cap

In [22]:
fig_vrds = viewer.plot_force_cot_theta_contour(
    load_case=load_case,
    n_axial=40,
    n_cot=40,
    metric="v_rd_s",
    show=False,
)
fig_vrds

In [23]:
fig_vrdmax = viewer.plot_force_cot_theta_contour(
    load_case=load_case,
    n_axial=40,
    n_cot=40,
    metric="v_rd_max",
    show=False,
)
fig_vrdmax

## 7. Uncracked V_Rd,c

Pass `use_uncracked_V_Rd_c=True` to use the Orr Eq.17 principal-stress based
concrete capacity as the V_Rd,c reference line.

In [24]:
fig_uncracked = viewer.plot_cot_theta_study(
    load_case=load_case,
    n_points=60,
    use_uncracked_V_Rd_c=True,
    title="Circular Shear Study — Uncracked V_Rd,c (Orr Eq.17)",
    show=False,
)
fig_uncracked

## 8. Axial-Moment Utilization with cot(theta) Slider


In [25]:
fig_mn_slider = viewer.plot_axial_moment_contour(
    V_Ed=load_case.V_Ed,
    loadcases=[(load_case.M_Ed, load_case.N_Ed)],
    n_moment=50,
    n_axial=50,
    n_cot=50,
    show=False,
)
fig_mn_slider


C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\analysis\circular_shear_viewer.py:123: UserWarning:

Effective depth fallback (no compression/tension split): d = 540.0 mm

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:650: UserWarning:

Lever arm fallback to 351.0 mm (0.65d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:650: UserWarning:

Lever arm fallback to 293.0 mm (0.71d): unable to compute a meaningful tension/compression centroid lever arm for this strain state.

C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:650: UserWarning:

Lever arm fallback to 368.2 mm (0.68d): unable to compute a meaningful tension/compression centroid lever arm for this strain s